### Dependencies

In [ ]:
import random
from qdrant_client import QdrantClient

import instructor
import json
from pydantic import BaseModel, Field

from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client
import tiktoken

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

Load all data from Qdrant

In [5]:
all_points = []
next_offset = None
batch_size = 100

while True:
    points, next_offset = qdrant_client.scroll(
        collection_name = "Amazon-items-collection-01-hybrid-search",
        limit = batch_size,
        offset= next_offset,
        with_payload = True,
        with_vectors = False
    )
    all_points.extend(points)
    if next_offset is None:
        break

In [6]:
len(all_points)

1000

In [ ]:
all_points

### Split the data into 2 groups: 50 items for synthetic question generation, 950 items to loop through and evaluate their relevance for previously generated synthetic questions.

In [8]:
rng = random.Random(42)
indices = list(range(len(all_points)))

In [10]:
rng.shuffle(indices)

In [ ]:
indices

In [12]:
sample_idx = set(indices[:50])

In [ ]:
sample_idx

In [15]:
sample_50 = [p for i, p in enumerate(all_points) if i in sample_idx]

In [ ]:
sample_50

In [18]:
remaining_points = [p for i, p in enumerate(all_points) if i not in sample_idx]

In [ ]:
remaining_points

### Generate synthetic questions

In [22]:
all_context_sample_50 = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in sample_50]

In [ ]:

all_context_sample_50

In [24]:
all_context_remaining_points = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in remaining_points]

In [ ]:
all_context_remaining_points

In [27]:
SYSTEM_PROMPT = f"""
You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant products and grounding its answers in them.

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that will eventually contain 1000 products.

## Your task
Generate exactly 30 questions that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** — answerable using exactly ONE product.
2. **Multi-product (10 questions)** — answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** — plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's point of view, in natural language.
- Refer to the items as "products". Never use the word "chunk".
- Keep questions specific. Even in the full 1000-product collection, a good question should be answerable from at most 4 products. Avoid broad or generic questions that a large number of products could satisfy.

## Products
{json.dumps(all_context_sample_50, indent=2)}
"""

In [ ]:
print(SYSTEM_PROMPT)

### Add Instructor for structured output

In [33]:
client = instructor.from_provider(
    "openai/gpt-5.4",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [ ]:
#for reference of below models
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            }
        },
    },
}

In [ ]:
#same purpose as output_schema
class SyntheticQuestion(BaseModel):
    reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
    question: str = Field(description="Suggested question.")
    chunk_ids: list[str] = Field(description="ID of the chunk that could be used to answer the question.")
    answer_example: str = Field(description="Suggested answer grounded in the context.")

class SyntheticQuestions(BaseModel):
    synthetic_questions: list[SyntheticQuestion] = Field(description="List of synthetic questions")

In [34]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT}
    ],
    reasoning={"effort": "none"},
    response_model=SyntheticQuestions
)

In [44]:
response.synthetic_questions

[SyntheticQuestion(reasoning='This question is answerable from the Hisense Roku TV product because it explicitly mentions compatibility with both Alexa and Google Assistant.', question='Which product is a 58-inch 4K Roku smart TV that works with both Alexa and Google Assistant?', chunk_ids=['B08L617VS9'], answer_example='The Hisense 58R6E3 is a 58-inch 4K Roku Smart TV and it works with both Alexa and Google Assistant.'),
 SyntheticQuestion(reasoning='This is answerable from the Samsung 990 PRO SSD listing, which specifies 1TB capacity, PCIe Gen4 x4 NVMe, and speeds up to 7,450/6,900 MB/s.', question='Do you have a 1TB internal SSD for a desktop or laptop with PCIe 4.0 NVMe speeds around 7,450 MB/s read?', chunk_ids=['B0C6N3GWW6'], answer_example='Yes, the Samsung 990 PRO 1TB is an M.2 PCIe Gen4 x4 NVMe internal SSD with sequential read/write speeds up to 7,450/6,900 MB/s.'),
 SyntheticQuestion(reasoning='The iPad keyboard case listing clearly states compatibility with iPad 9th/8th/7th

In [41]:
len(response.synthetic_questions)

31

In [42]:
for s_question in response.synthetic_questions:
    print(s_question.chunk_ids)

['B08L617VS9']
['B0C6N3GWW6']
['B08XMNYYWR']
['B0CJ4QWXZ5']
['B0BD5TK2MF']
['B0BLBRCMZ4']
['B0BTHTF4GB']
['B09PL8V7HS']
['B09WZDKSF7']
['B0B3ZKBJ8V']
['B0C8592NWX']
['B0B5L31SQT']
['B0B4BJ8YQ6']
['B0CDW92GPG']
['B0CDQ4NVQP']
['B09X9838WY', 'B0BC38F1DL', 'B0B3ZKBJ8V']
['B0BG7S3PCK', 'B09P4QW5Y2', 'B0BKZ7QG61']
['B09V7VC1LB', 'B0C61YP5Z5']
['B0BVDKD6JD', 'B09GK8LBWS']
['B0B3XWP4PM', 'B0B5X74W9P']
['B09V28H1J2', 'B09XGTMY6X']
['B0BWD5TKNH', 'B0BGR4TDW4']
['B08L617VS9', 'B09X2G3JZK']
['B0CJ4QWXZ5', 'B0BD5TK2MF']
['B0BVDKD6JD', 'B09GK8LBWS']
['B0C6GLTGQQ', 'B0CDW92GPG']
[]
[]
[]
[]
[]


In [45]:

synthetic_questions_relevant_data = [{"question_id": i, "question": item.question} for i, item in enumerate(response.synthetic_questions)]

In [46]:
synthetic_questions_relevant_data

[{'question_id': 0,
  'question': 'Which product is a 58-inch 4K Roku smart TV that works with both Alexa and Google Assistant?'},
 {'question_id': 1,
  'question': 'Do you have a 1TB internal SSD for a desktop or laptop with PCIe 4.0 NVMe speeds around 7,450 MB/s read?'},
 {'question_id': 2,
  'question': "I'm looking for a keyboard case for my 10.2-inch iPad 9th generation with a detachable backlit keyboard and pencil holder. Do you have one?"},
 {'question_id': 3,
  'question': 'Is there an indoor security camera with 360-degree motion tracking, privacy mode, and two-way audio?'},
 {'question_id': 4,
  'question': 'Do you sell a wall charger that secretly includes a WiFi camera for home monitoring?'},
 {'question_id': 5,
  'question': 'I need a removable privacy screen filter for a 15.6-inch laptop that also reduces blue light. What product fits that?'},
 {'question_id': 6,
  'question': 'Do you have a portable triple-screen monitor extender for a laptop that works with one USB-C ca

### Assign remaining chunks to relevant questions

In [79]:
SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products
{json.dumps(all_context_remaining_points[0:50], indent=2)}
"""

In [ ]:
print(SYSTEM_PROMPT_950)

### Count the number of tokens for the static prefix

In [47]:
SYSTEM_PROMPT_STATIC_PART = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products
"""

In [48]:
def count_tokens(text):
    encoding = tiktoken.get_encoding("o200k_base")

    return len(encoding.encode(text))

In [59]:
print(count_tokens(SYSTEM_PROMPT_STATIC_PART))

1378


### Assign chunks to relevant questions for a single batch

In [50]:
client = instructor.from_provider(
    "openai/gpt-5.4-mini",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [51]:
class SyntheticQuestionMatchingChunk(BaseModel):
    reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
    question_id: int = Field(description="ID of the question.")
    chunk_ids: list[str] = Field(description="IDs of the chunks that could be used to answer the question.")

class SyntheticQuestionMatchingChunks(BaseModel):
    synthetic_question_matching_chunks: list[SyntheticQuestionMatchingChunk] = Field(description="List of synthetic questions and their matching chunks")

In [56]:
#SYSTEM_PROMPT_STATIC_PART (part of  SYSTEM_PROMPT_950) is exactly the same every time, then that prefix is eligible for caching
tmp_response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT_950}
    ],
    reasoning={"effort": "none"},
    response_model=SyntheticQuestionMatchingChunks
)

In [ ]:
raw_response.usage

In [ ]:
tmp_response.synthetic_question_matching_chunks

In [ ]:
for data in tmp_response.synthetic_question_matching_chunks:
    print(f"Question ID: {data.question_id}, Chunk IDs: {data.chunk_ids}")


### ground-truth labels
##### Same as above just in one module

In [60]:
def get_relevant_chunks(synthetic_questions, remaining_points_batch):

    SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions, indent=2)}

## Products
{json.dumps(remaining_points_batch, indent=2)}
"""

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_950}
        ],
        reasoning={"effort": "none"},
        response_model=SyntheticQuestionMatchingChunks
    )

    return response.synthetic_question_matching_chunks

In [ ]:
answer_extended_chunks = []

#running in interval of 50
for i in range(0, len(all_context_remaining_points), 50):

    print(f"Working on batch {(i+50)/50}")

    answer_extended_chunks.append(get_relevant_chunks(synthetic_questions_relevant_data, all_context_remaining_points[i:i+50]))

In [ ]:
answer_extended_chunks

In [ ]:
for i, data in enumerate(answer_extended_chunks):
    print(f"Batch {i+1}")
    for item in data:
        print(f"Question ID: {item.question_id}, Chunk IDs: {item.chunk_ids}")

In [75]:
questions_container = {i: [] for i in range(len(synthetic_questions_relevant_data))}

In [ ]:
questions_container

In [ ]:
for i, data in enumerate(answer_extended_chunks):
    print(f"Batch {i+1}")
    for item in data:
        questions_container[item.question_id].extend(item.chunk_ids)

In [78]:
questions_container

{0: [],
 1: [],
 2: ['B09XTJ97BR', 'B0BCC2QRJ3', 'B09PTX6461'],
 3: ['B0BZ44Y4C6', 'B09MJVHH65'],
 4: ['B0C6PKB5QW', 'B0BV6YQHHH'],
 5: ['B0CC96MBWL'],
 6: ['B0CC96MBWL'],
 7: [],
 8: ['B09TFM1SFQ', 'B0BQJ5H3H2', 'B0C13M8GJZ'],
 9: ['B09ZK2QWD1'],
 10: ['B0C9ZYXHHB'],
 11: [],
 12: [],
 13: ['B0C4YQ74TX', 'B0B2W2LXZN'],
 14: ['B0CD7CC716', 'B0C4Z4S6WT', 'B0BFVVFJG3', 'B09WVZFTTK', 'B0B6DNL999'],
 15: ['B0C5M2SLKL',
  'B09YCHGW1B',
  'B0BRV544MV',
  'B0BYSP3HT9',
  'B09WGLRZFP',
  'B0BTYFJRZQ',
  'B0BB7JD7DD',
  'B0BYYP3NS7',
  'B0BRMV85ZK',
  'B0CB3CLQYS',
  'B09XTDJBDM',
  'B0BRTRHQKX',
  'B0BC4PGXFK',
  'B0BGC9N61V',
  'B0BXW87PHC',
  'B0BQTZ7TDP',
  'B0BWJB13Z3',
  'B09TFM1SFQ',
  'B0C2Q17PND',
  'B09KZ7G4NW',
  'B0BK8WW5DH',
  'B0BRN967WJ',
  'B0C7HY8NKW',
  'B0BX8XW9CH',
  'B0BGPK4MX9',
  'B0C3C9KLD2',
  'B0BPS7JXPN',
  'B0BMPS5V6H',
  'B0BD8FXHPB',
  'B0BPS7JXPN',
  'B0BPS7JXPN',
  'B0BPS7JXPN',
  'B0BM93SM2P',
  'B0BRH3JTWJ',
  'B09Q5HYKCY',
  'B0BRXDRGDP',
  'B0B62R4CCJ',
  'B0

### Add response.synthetic_questions chunks to questions_container, as they were 50 items we took for synthetic questions

In [80]:
for i , data in enumerate(response.synthetic_questions):
    questions_container[i].extend(data.chunk_ids)

In [81]:
questions_container

{0: ['B08L617VS9'],
 1: ['B0C6N3GWW6'],
 2: ['B09XTJ97BR', 'B0BCC2QRJ3', 'B09PTX6461', 'B08XMNYYWR'],
 3: ['B0BZ44Y4C6', 'B09MJVHH65', 'B0CJ4QWXZ5'],
 4: ['B0C6PKB5QW', 'B0BV6YQHHH', 'B0BD5TK2MF'],
 5: ['B0CC96MBWL', 'B0BLBRCMZ4'],
 6: ['B0CC96MBWL', 'B0BTHTF4GB'],
 7: ['B09PL8V7HS'],
 8: ['B09TFM1SFQ', 'B0BQJ5H3H2', 'B0C13M8GJZ', 'B09WZDKSF7'],
 9: ['B09ZK2QWD1', 'B0B3ZKBJ8V'],
 10: ['B0C9ZYXHHB', 'B0C8592NWX'],
 11: ['B0B5L31SQT'],
 12: ['B0B4BJ8YQ6'],
 13: ['B0C4YQ74TX', 'B0B2W2LXZN', 'B0CDW92GPG'],
 14: ['B0CD7CC716',
  'B0C4Z4S6WT',
  'B0BFVVFJG3',
  'B09WVZFTTK',
  'B0B6DNL999',
  'B0CDQ4NVQP'],
 15: ['B0C5M2SLKL',
  'B09YCHGW1B',
  'B0BRV544MV',
  'B0BYSP3HT9',
  'B09WGLRZFP',
  'B0BTYFJRZQ',
  'B0BB7JD7DD',
  'B0BYYP3NS7',
  'B0BRMV85ZK',
  'B0CB3CLQYS',
  'B09XTDJBDM',
  'B0BRTRHQKX',
  'B0BC4PGXFK',
  'B0BGC9N61V',
  'B0BXW87PHC',
  'B0BQTZ7TDP',
  'B0BWJB13Z3',
  'B09TFM1SFQ',
  'B0C2Q17PND',
  'B09KZ7G4NW',
  'B0BK8WW5DH',
  'B0BRN967WJ',
  'B0C7HY8NKW',
  'B0BX8XW9CH',
  '

### get descriptions for above id's

In [82]:
def get_description(parent_asin: str) -> str:
    
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01-hybrid-search",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]

    return points[0].payload["preprocessed_description"]

In [83]:
get_description("B0C6N3GWW6")

"Samsung 990 PRO Series - 1TB PCIe Gen4. X4 NVMe 2.0c - M.2 Internal SSD (MZ-V9P1T0B/AM) ADOBE MEMBERSHIP: Get a two-month membership of Adobe Creative Cloud Photography plan on us when you purchase and register an eligible 1TB or 2TB Samsung SSD* HUGE SPEED BOOST: Get random read/write speeds that are 40%/55% faster than 980 PRO; Experience up to 1400K/1550K IOPS, while sequential read/write speeds up to 7,450/6,900 MB/s reach near the max performance of PCIe 4.0* BREAKTHROUGH POWER EFFICIENCY: Use less power and get more performance; Enjoy up to 50% improved performance per watt over 980 PRO, plus optimal power efficiency with max PCIe 4.0 performance** SMART THERMAL CONTROL: Samsung's own nickel-coated controller delivers effective thermal control; With its slim size, 990 PRO is a perfect fit for desktops and laptops that meet the PCI-SIG D8 standard*** THE CHAMPION MAKER: Up to 65% improvement in random performance enables faster loads for an ultimate gaming experience on PS5 and D

In [86]:
def generate_golden_answer(question, relevant_chunks):

    class GoldenAnswer(BaseModel):
        answer: str = Field(description="The ideal reference answer for the question.")

    PROMPT = f"""
You are the shopping assistant of an e-shop. You are also being used to generate the ideal reference answer for a single customer question, to serve as ground truth for evaluating a RAG system.

## Context
A customer asked a question. A retrieval system has already gathered the products most relevant to it. Your job is to write the best possible answer a helpful shopping assistant could give, using ONLY those products.

## Inputs
- `question`: the customer's question.
- `products`: a list of relevant products, each with a `text` description. This may be empty or may not actually contain the answer.

## How to write the answer
- Ground every factual claim strictly in the provided product descriptions. Never invent products, prices, specs, availability, or details that are not present in the text.
- Answer in the voice of a friendly, concise shopping assistant speaking directly to the customer.
- Use only the information needed to answer well; don't dump every product if only some are relevant.
- If the products only partially answer the question, answer what you can and clearly state what information isn't available.
- If the products do not contain the information needed to answer at all (or the list is empty), do not guess. Respond the way a good assistant would when the shop doesn't carry the item or the info isn't available (e.g. politely say you couldn't find it), and do not fabricate an answer.

## Question
{json.dumps(question, indent=2)}

## Products
{json.dumps(relevant_chunks, indent=2)}"""

    response, raw_response = client.create_with_completion(
            messages=[
                {"role": "system", "content": PROMPT}
            ],
            reasoning={"effort": "none"},
            response_model=GoldenAnswer
    )

    return response.answer


In [ ]:
reference_dataset = []

for i, data in enumerate(response.synthetic_questions):

    print(f"Working on question {i+1}")

    # synthetic questions has same index as questions_container. i.e 1st synthetic question == question_container[0]
    relevant_chunks = [get_description(item) for item in questions_container[i]]

    golden_answer = generate_golden_answer(data.question, relevant_chunks)

    reference_dataset.append({
        "question": data.question,
        "ground_truth": golden_answer,
        "reference_context_ids": questions_container[i],
        "reference_descriptions": relevant_chunks
    })

In [88]:
reference_dataset

[{'question': 'Which product is a 58-inch 4K Roku smart TV that works with both Alexa and Google Assistant?',
  'ground_truth': 'The product you’re looking for is the Hisense 58R6E3 58-Inch Model 2020 4K Roku Smart TV. It’s a 58-inch 4K Roku smart TV and it works with both Alexa and Google Assistant.',
  'reference_context_ids': ['B08L617VS9'],
  'reference_descriptions': ['Hisense 58R6E3 58-Inch Model 2020 4K Roku Smart TV LED Compatibility with Google Assistant and Alexa, Motion Rate 120, HDR10, DTS Studio Sound 4K Ultra HD (2160p resolution): Enjoy stunning 4K movies and TV shows with 4x the resolution of Full HD and upscale your current content to Ultra HD-level picture quality Smart TV with access to streaming services for entertainment: Enjoy instant access to the best selection of apps from the best streaming services like Netflix, Disney+, YouTube and many more right on your TV Works with Alexa and Google Assistant: Ask the Google Assistant or Alexa to turn on your TV, change c

### Create Langsmith dataset

In [89]:
ls_client = Client()

In [90]:
dataset_name = "rag-evaluation-dataset-extended"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset for 1000 items"
)

In [91]:
for item in reference_dataset:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"]
        },
        outputs={
            "ground_truth": item["ground_truth"],
            "reference_context_ids": item["reference_context_ids"],
            "reference_descriptions": item["reference_descriptions"]
        }
    )